# 03 — ASR: Whisper-large-v3 Fine-tuning with LoRA

Fine-tunes `openai/whisper-large-v3` on ViMD using **LoRA/QLoRA** (4-bit quantization) to fit in Colab T4 16GB.

**Target KPIs**: WER ≤ 8% | CER ≤ 4%


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = '/content/drive/MyDrive/VietSpeech'
DATASET_PATH = f'{PROJECT_ROOT}/hf_dataset'
CHECKPOINT_DIR = f'{PROJECT_ROOT}/checkpoints/asr'
EXPORT_DIR = f'{PROJECT_ROOT}/weights/asr'

import sys, os
sys.path.insert(0, PROJECT_ROOT)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(EXPORT_DIR, exist_ok=True)

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
!pip install -q transformers datasets accelerate evaluate peft bitsandbytes
!pip install -q jiwer soundfile mlflow
print('Dependencies installed')

In [ ]:
# ── Load dataset ─────────────────────────────────────────────────────────
from datasets import load_from_disk, Audio

ds = load_from_disk(DATASET_PATH).cast_column('audio', Audio(sampling_rate=16000))
print('Dataset loaded:', ds)

In [ ]:
# ── Load Whisper + Processor ──────────────────────────────────────────────
from transformers import WhisperProcessor, WhisperForConditionalGeneration, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

MODEL_NAME = 'openai/whisper-large-v3'

processor = WhisperProcessor.from_pretrained(MODEL_NAME, language='Vietnamese', task='transcribe')

# 4-bit quantization config for T4
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
)
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []

print(f'Whisper-large-v3 loaded (4-bit)')

In [ ]:
# ── Apply LoRA ────────────────────────────────────────────────────────────
lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=['q_proj', 'v_proj', 'k_proj', 'out_proj', 'fc1', 'fc2'],
    lora_dropout=0.1,
    bias='none',
    task_type=TaskType.SEQ_2_SEQ_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Expected: ~10-15M trainable / 1.5B total

In [ ]:
# ── Preprocessing ─────────────────────────────────────────────────────────
import numpy as np

def preprocess_whisper(batch):
    audio = [a['array'] for a in batch['audio']]
    features = processor.feature_extractor(
        audio, sampling_rate=16000, return_tensors='pt'
    )
    batch['input_features'] = features.input_features
    labels = processor.tokenizer(
        batch['transcript'], return_tensors='pt', padding=True, truncation=True, max_length=448
    )
    batch['labels'] = labels.input_ids
    return batch

ds_proc = ds.map(preprocess_whisper, batched=True, batch_size=8, num_proc=1,
                 remove_columns=['audio'])
ds_proc.set_format(type='torch', columns=['input_features', 'labels'])
print('Preprocessing done:', ds_proc)

In [ ]:
# ── Training ──────────────────────────────────────────────────────────────
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import evaluate

metric = evaluate.load('wer')

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{'input_features': f['input_features']} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors='pt')
        label_features = [{'input_ids': f['labels']} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors='pt')
        labels = labels_batch['input_ids'].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch['labels'] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.batch_decode(label_ids, skip_special_tokens=True)
    wer_score = 100 * metric.compute(predictions=pred_str, references=label_str)
    return {'wer': wer_score}

training_args = Seq2SeqTrainingArguments(
    output_dir=CHECKPOINT_DIR,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    warmup_steps=200,
    max_steps=5000,
    fp16=True,
    evaluation_strategy='steps',
    eval_steps=500,
    save_steps=500,
    load_best_model_at_end=True,
    metric_for_best_model='wer',
    greater_is_better=False,
    predict_with_generate=True,
    generation_max_length=448,
    logging_steps=50,
    report_to=['mlflow'],
    push_to_hub=False,
    dataloader_num_workers=2,
    remove_unused_columns=False,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=ds_proc['train'],
    eval_dataset=ds_proc['val'],
    tokenizer=processor.feature_extractor,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()
print('\nASR Training complete!')

In [ ]:
# ── Save merged model (LoRA merged into base weights) ─────────────────────
from peft import PeftModel

# Merge LoRA weights into base model for efficient inference
model_merged = model.merge_and_unload()
model_merged.save_pretrained(EXPORT_DIR)
processor.save_pretrained(EXPORT_DIR)

print(f'Merged ASR model saved to: {EXPORT_DIR}')
print(f'Files: {os.listdir(EXPORT_DIR)}')

In [ ]:
# ── Quick evaluation on test set ──────────────────────────────────────────
print('Running test set evaluation...')
test_results = trainer.predict(ds_proc['test'])
test_wer = compute_metrics(test_results)
print(f'Test WER: {test_wer["wer"]:.2f}%')
print(f'Target: ≤ 8.0%  → {"PASS" if test_wer["wer"] <= 8.0 else "FAIL"}')

## ✓ Whisper ASR Fine-tuning Complete

**Model saved to**: `{EXPORT_DIR}/`

**To use locally after training**:
1. Copy the `VietSpeech/weights/asr/` folder from Google Drive to `storage/weights/asr/`
2. Set `ASR_MODEL_PATH=storage/weights/asr` in your `.env`
3. `WhisperInference.load()` will load the merged model automatically

**Next**: Run `04_asr_phowhisper.ipynb`